# Producer Pattern 5 of 5 — Resilient Kafka Producer & Dead-Letter Recovery
### Beginner → failure reasoning → retries & circuit breaking → durable outbox → production-style implementation → offline tests

**Business story:** An e-commerce service accepts an order. It must publish an `order.created` event, even if Kafka becomes slow or temporarily unavailable. What prevents the event from disappearing? What happens after repeated failure?

**One-line mental model:** **save durably → submit asynchronously → process acknowledgement → retry with limits → quarantine failures → recover deliberately**.

| Notebook | Pattern | Main takeaway |
|---|---|---|
| 1 | Fire-and-forget | `produce()` queues messages; delivery is asynchronous |
| 2 | Callback-confirmed | Delivery callbacks reveal final success/failure |
| 3 | Shared producer | Reuse a producer per application process/configuration |
| 4 | Content-based router | Decide **where** the event goes |
| **5 — this notebook** | **Resilient producer / DLQ** | **Decide what to do when delivery does not succeed** |

**Important reality check:** A Kafka DLQ is another Kafka topic. **If the same Kafka cluster is unreachable, writing to its DLQ will also fail.** To avoid losing important events during that outage, we introduce a **transactional outbox** in a durable database. A DLQ is for *quarantining records that require separate handling*, not magically bypassing a broken broker.

**Audience:** Junior developers. We introduce each term before using it, use standard-library Python wherever possible, show the minimal idea first, and then build a reusable implementation. Run everything offline; live Kafka is **opt-in**.

## Learning objectives (read this once)

| Section | Question answered | What you retain |
|---|---|---|
| 0 | What do I need installed? | Offline first, live broker optional |
| 1 | What kinds of failures exist? | Queue rejection ≠ delivery failure ≠ unknown outcome |
| 2 | What does Kafka retry automatically? | Don't rebuild librdkafka's internal retry engine |
| 3 | How do I retry safely? | Bounded exponential backoff, then stop |
| 4 | What is a circuit breaker? | Stop hammering an unhealthy dependency |
| 5 | What is a DLQ? | Quarantine poison/exhausted messages; Kafka DLQ needs Kafka |
| 6 | What if the broker is down? | Persist first in a transactional outbox |
| 7 | How do leases/acknowledgements work? | Work is pending until broker delivery is confirmed |
| 8 | How do patterns fit together? | One relay, shared producer, callbacks, retry/DLQ |
| 9 | Can this be tested without Kafka? | Deterministic fake producer and clock |
| 10 | Does the implementation handle edge cases? | Offline tests, including crashes and outages |
| 11 | How do I run a real broker experiment? | Explicit opt-in; no destructive topic recreation |
| 12 | Where would this live in FastAPI? | Database transaction + background relay ownership |
| 13 | What should I monitor? | Backlog, oldest age, failures, DLQ, circuit state |
| 14 | What are the production boundaries? | At-least-once, deduplication, transactions, scale |
| 15 | What do engineers remember? | One-page checklist and decision tree |
| 16 | What should I practise? | Practical tasks and failure drills |

---
# Section 0 — Safe setup

**Objective:** Learn the design without depending on a broker or a cloud account.

This notebook's database is a temporary SQLite file, used as a teaching tool. **In a distributed production service, use your application's durable transactional database** (for example, PostgreSQL or Azure SQL) for its outbox. A local SQLite file in an ephemeral container does not provide durable recovery after the container is deleted.

Live publishing requires `confluent-kafka` and Kafka at `localhost:9092`; install only if you intend to use that section:

```python
%pip install confluent-kafka
```

`RUN_LIVE_KAFKA` defaults to `False`. The offline labs require only Python's standard library.

In [ ]:
import json
import logging
import os
import random
import sqlite3
import tempfile
import threading
import time
import uuid
from collections import deque
from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Optional

RUN_LIVE_KAFKA = False
BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
ORDERS_TOPIC = "lab5.orders.created"
DLQ_TOPIC = "lab5.orders.dead_letter"

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
log = logging.getLogger("resilient_producer")

print("Mode:", "LIVE" if RUN_LIVE_KAFKA else "OFFLINE")
print("Broker (only for live mode):", BOOTSTRAP)

---
# Section 1 — Begin with the smallest possible failure story

**Objective:** Stop treating every failure as the same thing.

Think of a courier desk:

1. **Desk is full:** It cannot accept your package **right now** → local `BufferError` from `produce()`.
2. **Package was accepted, but delivery ultimately failed:** You receive a failure slip → non-`None` error in `on_delivery` (after Kafka's own attempts).
3. **You closed the office before hearing back:** Your `flush(timeout)` returned with outstanding work → **unknown final outcome**. Do not claim it succeeded or failed.
4. **The shipping label itself is invalid:** Reject it before submitting; repeated retries will not fix the data.

```text
Application creates event
          |
          v
   Validate/serialize ---- invalid ----> quarantine/reject
          |
          v
    produce() attempt ---- BufferError -> bounded wait/retry
          |
          v
    Accepted locally
          |
          v
    Background networking
          |
          +--- delivery success ---> mark event delivered
          |
          +--- final delivery error ---> retry later OR dead-letter
          |
          +--- no callback yet ---> keep durable event pending
```

**Words to remember:** *synchronous queue rejection*, *asynchronous delivery failure*, and *ambiguous/unknown outcome* are different states.

## 1.1 A minimal delivery callback

**Objective:** Reuse Notebook 2 knowledge; the callback is **not itself a retry loop**.

The producer sends in its background networking threads. `poll()`/`flush()` services the delivery callback. Below is a reference snippet, not a cell that runs offline:

```python
producer.produce(
    topic="orders.created",
    key="order-101",
    value=b'{"order_id":"order-101"}',
    on_delivery=on_delivery,
)
producer.poll(0)   # process ready reports; not a send command
```

If `on_delivery(error, message)` receives an error, that is the **final outcome of this producer delivery attempt**, potentially *after* retries already performed inside `librdkafka`.

---
# Section 2 — Do not confuse Kafka's internal retries with your application's retries

**Objective:** Choose which layer is responsible for recovery.

| Mechanism | Who owns it? | Example | Should your code retry manually? |
|---|---|---|---|
| Broker/network retries | `librdkafka` | Transient disconnected socket | Usually **no**; configured producer handles these automatically |
| Local queue pressure | Your publishing code | `BufferError` from `produce()` | Yes, bounded re-attempt to **enqueue** |
| Final failed delivery | Your recovery workflow | Delivery callback reports timeout | Maybe; retry **the business event** with policy and understand duplicates |
| Invalid business event | Your validation layer | Missing required order ID | **No**; fix or quarantine |
| Outage / uncertain result | Durable outbox workflow | Process crashes after submitting | Recover by replay, expect possible duplicates |

**Do not** implement an infinite manual retry loop inside the delivery callback. Do not call `flush()` after every message. Do not assume `error.retriable()` alone decides whether a failed business event is safe to replay: a timeout can have an *ambiguous outcome* even if the final callback says delivery failed.

### Why a stable event ID matters

If a callback reports a timeout, the broker **might** have stored the message while the producer failed to receive confirmation. Replaying it can produce a second record. Use a stable `event_id` when retrying and make consumers **idempotent** (able to ignore previously processed IDs).

`enable.idempotence=True` helps deduplicate eligible retries **within a Kafka producer session**. It does **not** guarantee deduplication when your application recreates the event in a new session after a crash.

---
# Section 3 — Bounded exponential backoff with jitter

**Objective:** Avoid sending retry traffic in a tight loop.

If something keeps failing, immediate retry produces unnecessary pressure. Instead wait increasingly longer:

```text
failed attempt #1 -> wait ~1 second
failed attempt #2 -> wait ~2 seconds
failed attempt #3 -> wait ~4 seconds
... capped at configured maximum
```

**Jitter** adds randomness so that 1,000 application instances do not all retry at exactly the same millisecond. The code below uses **full jitter**: a random delay between 0 and the exponential cap.

**Important:** Here, `attempt` means an **application-level outbox dispatch attempt**, *not* Kafka's internal packet retries.

In [ ]:
@dataclass(frozen=True)
class RetryPolicy:
    """Calculate delays; scheduling is handled elsewhere."""

    max_attempts: int = 3
    initial_delay_seconds: float = 1.0
    max_delay_seconds: float = 30.0

    def __post_init__(self):
        if self.max_attempts < 1:
            raise ValueError("max_attempts must be >= 1")
        if self.initial_delay_seconds <= 0 or self.max_delay_seconds <= 0:
            raise ValueError("retry delays must be positive")

    def delay(self, attempt: int, random_fraction: float = 0.5) -> float:
        """Return a capped exponential delay with full jitter.

        attempt=1 means the first delivery attempt just failed.
        random_fraction can be fixed in tests for repeatability.
        """
        if attempt < 1 or not (0.0 <= random_fraction <= 1.0):
            raise ValueError("invalid retry attempt or random_fraction")

        cap = min(self.max_delay_seconds, self.initial_delay_seconds * 2 ** (attempt - 1))
        return cap * random_fraction


retry_policy = RetryPolicy(max_attempts=4, initial_delay_seconds=1, max_delay_seconds=8)
for attempt in range(1, 5):
    print(f"After attempt {attempt}: wait {retry_policy.delay(attempt, 0.5):.1f}s (sample)")

assert retry_policy.delay(4, 1.0) == 8.0

**Why only four attempts?** Four is a **demonstration configuration**, not an industry-wide magic number. In production, choose retry budgets using message criticality, delivery timeout, outage expectations, and backlog capacity. A retry budget must be finite, but critical events shouldn't be discarded merely because the budget expired; they need a durable recovery path.

---
# Section 4 — Circuit breaker: stop hammering an unhealthy dependency

**Objective:** Understand the protection mechanism separately from Kafka publishing.

**Analogy:** The courier desk has reported repeated delivery failures. Instead of giving it new packages indefinitely, you temporarily close intake. After a cooldown, you allow **one test package**. If it works, reopen; otherwise, remain closed.

Three states:

- **CLOSED (normal):** New sends permitted; track failures.
- **OPEN:** New sends temporarily denied; the **outbox still holds the events**.
- **HALF_OPEN:** Cooldown elapsed; permit one probe to see whether Kafka recovered.

**Important:** Circuit breakers decide whether to attempt *new* submissions; they cannot cancel messages already accepted into the producer. `BufferError` alone is not proof Kafka is down.

In [ ]:
class CircuitBreaker:
    """Small thread-safe circuit breaker for NEW dispatch attempts.

    A `ticket` returned by allow() is either 'normal' or 'probe'.
    Pass that ticket to success(), failure(), or cancel().
    """

    def __init__(self, threshold=3, cooldown_seconds=5.0, clock=time.monotonic):
        if threshold < 1 or cooldown_seconds <= 0:
            raise ValueError("invalid breaker configuration")
        self.threshold = threshold
        self.cooldown_seconds = cooldown_seconds
        self.clock = clock
        self.state = "CLOSED"
        self.failures = 0
        self.open_until = 0.0
        self.probe_in_flight = False
        self.lock = threading.Lock()

    def allow(self):
        """Return a permit ticket, or None if no new attempt is allowed."""
        with self.lock:
            if self.state == "OPEN":
                if self.clock() < self.open_until:
                    return None
                self.state = "HALF_OPEN"

            if self.state == "HALF_OPEN":
                if self.probe_in_flight:
                    return None
                self.probe_in_flight = True
                return "probe"

            return "normal"

    def success(self, ticket):
        with self.lock:
            if ticket == "probe" and self.state == "HALF_OPEN":
                self.state, self.failures, self.probe_in_flight = "CLOSED", 0, False
            elif ticket == "normal" and self.state == "CLOSED":
                self.failures = 0

    def failure(self, ticket):
        with self.lock:
            if ticket == "probe" and self.state == "HALF_OPEN":
                self._open()
            elif ticket == "normal" and self.state == "CLOSED":
                self.failures += 1
                if self.failures >= self.threshold:
                    self._open()

    def cancel(self, ticket):
        """The send was never accepted (e.g. local queue full)."""
        with self.lock:
            if ticket == "probe" and self.state == "HALF_OPEN":
                self.probe_in_flight = False

    def _open(self):
        self.state = "OPEN"
        self.open_until = self.clock() + self.cooldown_seconds
        self.probe_in_flight = False


# This toy experiment is deterministic; no real waiting or Kafka is needed.
class FakeClock:
    def __init__(self):
        self.seconds = 1_000.0

    def monotonic(self):
        return self.seconds

    def wall(self):
        return self.seconds

    def advance(self, seconds):
        self.seconds += seconds


clock = FakeClock()
breaker_demo = CircuitBreaker(threshold=2, cooldown_seconds=5, clock=clock.monotonic)
for _ in range(2):
    permit = breaker_demo.allow()
    breaker_demo.failure(permit)
print("After two failures:", breaker_demo.state)
assert breaker_demo.allow() is None
clock.advance(5)
probe = breaker_demo.allow()
print("After cooldown:", breaker_demo.state, "| permit:", probe)
assert probe == "probe" and breaker_demo.allow() is None
breaker_demo.success(probe)
print("After successful probe:", breaker_demo.state)
assert breaker_demo.state == "CLOSED"

**Caveat:** This simplified breaker uses final delivery failures as its signal. A real application may use a rolling error rate, Kafka error classifications, and separate health metrics. Some permanent record errors (such as an invalid schema) are **not** evidence that the broker is unhealthy and should not open the breaker. The single-probe rule here applies to this breaker instance, not all running application replicas.

---
# Section 5 — Dead-Letter Queue (DLQ): what is it really?

**Objective:** Distinguish a *quarantine* from a *retry mechanism*.

**Analogy:** An undeliverable parcel goes to a **special investigation shelf**, with its label, its contents (subject to privacy controls), and the reason for failure. Someone can inspect, repair, and replay it later.

A DLQ record should answer: **Which event? Originally intended for which topic? What went wrong? How many times? When?**

```text
       Retry allowed                 Retry exhausted / invalid
PENDING ----------> DELIVERY FAILED -------------------------->
   ^                      |                                    DLQ
   |                      |
   +--------- wait/backoff+

When broker is DOWN:
  Kafka topic (including Kafka DLQ) may be UNREACHABLE.
  So retain the event in a DURABLE DATABASE until recovery.
```

**Nuance:** DLQs are especially common in **consumers**, where a record cannot be processed. For a **producer**, failed event delivery also needs recovery, but writing to a DLQ in the *same unreachable cluster* is not sufficient. In this notebook, **durable dead-letter storage is the source of truth**; a Kafka DLQ topic can be a downstream projection once Kafka recovers.

In [ ]:
def dead_letter_envelope(original_event: dict, *, topic: str, attempts: int, reason: str):
    """Describe a quarantined event without logging secrets."""
    return {
        "original_event_id": original_event["event_id"],
        "original_topic": topic,
        "attempts": attempts,
        "reason": reason,
        "quarantined_at": datetime.now(timezone.utc).isoformat(),
        "original_event": original_event,
    }

sample = {
    "event_id": "evt-demo-001",
    "event_type": "order.created",
    "order_id": "order-101",
}
print(json.dumps(dead_letter_envelope(sample, topic=ORDERS_TOPIC, attempts=3,
                                       reason="Delivery attempts exhausted"), indent=2))

**Security rule:** Do not copy payment card details, credentials, protected health information, or other sensitive fields blindly into a DLQ. Apply access controls, retention policy, and payload minimization/redaction.

---
# Section 6 — Transactional outbox: what survives an outage or crash?

**Objective:** Guarantee that an accepted order and its event are saved in **the same database transaction**.

Consider this fragile approach:

```python
save_order_to_database(order)  # succeeds
producer.produce(...)          # the process crashes here
```

The order exists, but the event may never be published. Reversing the calls is not safe either: the event might publish while the database transaction later fails.

**Outbox solution:** In ONE local database transaction, save the business order **and** an outbox row representing its event. A separate relay sends the outbox row to Kafka. The outbox row is marked delivered **only when Kafka's delivery callback reports success**.

```text
HTTP request
     |
     v
Database transaction
  +----------------------+--------------------------------+
  | orders table         | outbox table                   |
  | order-101 stored     | evt-001 PENDING               |
  +----------------------+--------------------------------+
                          |
                          v
                 Background outbox relay
                          |
                          v
                 Shared Kafka producer
                          |
             success callback / failure callback
                   |                 |
                   v                 v
             DELIVERED         PENDING or DEAD_LETTER
```

**What this guarantees (if the database itself is durable):** The order and its event are committed together or rolled back together. It does **not** make Kafka and the database one atomic transaction; therefore the relay provides **at-least-once** delivery and duplicates remain possible.

We will use SQLite to make this concept runnable without Kafka. In enterprise deployments, the database transaction would normally use your existing application database.

## 6.1 The durable outbox schema and repository

**Objective:** Give the relay a reliable source of pending work.

Fields you should recognize:

- `event_id`: stable unique identity, reused across retries.
- `status`: `pending`, `leased`, `delivered`, or `dead_letter`.
- `attempts`: number of accepted **dispatch attempts**. We undo the count if the local producer queue rejects the event.
- `next_attempt_at`: do not retry before this time.
- `lease_token` / `lease_until`: prevent two workers from claiming the same row simultaneously; expired leases permit crash recovery.
- `last_error`: a short failure reason for diagnosis.
- `dlq_published`: whether a dead-letter copy was confirmed in a Kafka DLQ topic; the database retains the original.

**Why leases?** If the relay crashes after claiming an event, a permanent `in_progress` flag would trap it forever. A lease expires, allowing a later worker to retry. The lease duration must exceed the maximum expected delivery processing time, or you need lease renewal; otherwise a slow in-flight message can be claimed again.

In [ ]:
@dataclass(frozen=True)
class OutboxRecord:
    event_id: str
    topic: str
    message_key: str
    payload_json: str
    attempts: int
    lease_token: str


class SQLiteOutbox:
    """Teaching implementation of the transactional outbox.

    Each operation opens its own SQLite connection so callbacks and
    workers do not share a sqlite3.Connection across threads.

    Replace with your application's transactional DB in production.
    """

    def __init__(self, db_path):
        self.db_path = str(db_path)
        if self.db_path == ":memory:":
            raise ValueError("Use a file-backed database for this lab")
        self._init_schema()

    @contextmanager
    def _connect(self):
        """Begin a transaction and close the connection after each operation."""
        conn = sqlite3.connect(self.db_path, timeout=10)
        try:
            conn.row_factory = sqlite3.Row
            conn.execute("PRAGMA busy_timeout = 10000")
            with conn:
                yield conn
        finally:
            conn.close()

    def _init_schema(self):
        with self._connect() as conn:
            conn.executescript("""
                CREATE TABLE IF NOT EXISTS orders (
                    order_id TEXT PRIMARY KEY,
                    customer_id TEXT NOT NULL,
                    amount_cents INTEGER NOT NULL CHECK(amount_cents > 0)
                );
                CREATE TABLE IF NOT EXISTS outbox (
                    event_id TEXT PRIMARY KEY,
                    topic TEXT NOT NULL,
                    message_key TEXT NOT NULL,
                    payload_json TEXT NOT NULL,
                    status TEXT NOT NULL DEFAULT 'pending'
                        CHECK(status IN ('pending','leased','delivered','dead_letter')),
                    attempts INTEGER NOT NULL DEFAULT 0,
                    next_attempt_at REAL NOT NULL DEFAULT 0,
                    lease_until REAL,
                    lease_token TEXT,
                    last_error TEXT,
                    dlq_published INTEGER NOT NULL DEFAULT 0
                );
                CREATE INDEX IF NOT EXISTS idx_outbox_due
                    ON outbox(status, next_attempt_at, lease_until);
            """)

    def create_order(self, *, order_id, customer_id, amount_cents, now=None):
        """Persist the order and its event atomically; NO Kafka call here."""
        if not order_id or not customer_id or not isinstance(amount_cents, int) or amount_cents <= 0:
            raise ValueError("order_id, customer_id and positive integer amount_cents required")
        event_id = str(uuid.uuid4())
        event = {
            "event_id": event_id,
            "event_type": "order.created",
            "order_id": order_id,
            "customer_id": customer_id,
            "amount_cents": amount_cents,
        }
        with self._connect() as conn:
            # These writes are in the SAME SQLite transaction.
            conn.execute("INSERT INTO orders VALUES (?, ?, ?)",
                         (order_id, customer_id, amount_cents))
            conn.execute("""
                INSERT INTO outbox
                    (event_id, topic, message_key, payload_json, next_attempt_at)
                VALUES (?, ?, ?, ?, ?)
            """, (event_id, ORDERS_TOPIC, order_id, json.dumps(event),
                  time.time() if now is None else now))
        return event_id

    def claim_due(self, *, now, lease_seconds=60.0):
        """Claim ONE event atomically; also reclaim expired leases.

        Returns None if no event is ready.
        """
        if lease_seconds <= 0:
            raise ValueError("lease_seconds must be > 0")
        with self._connect() as conn:
            conn.execute("BEGIN IMMEDIATE")
            row = conn.execute("""
                SELECT * FROM outbox
                WHERE (status='pending' AND next_attempt_at <= ?)
                   OR (status='leased' AND lease_until <= ?)
                ORDER BY next_attempt_at, rowid
                LIMIT 1
            """, (now, now)).fetchone()
            if row is None:
                return None

            token = str(uuid.uuid4())
            attempt_number = row["attempts"] + 1
            conn.execute("""
                UPDATE outbox SET status='leased', attempts=?,
                       lease_until=?, lease_token=?
                WHERE event_id=?
            """, (attempt_number, now + lease_seconds, token, row["event_id"]))
            return OutboxRecord(
                event_id=row["event_id"], topic=row["topic"],
                message_key=row["message_key"], payload_json=row["payload_json"],
                attempts=attempt_number, lease_token=token,
            )

    def quarantine_expired(self, *, now, max_attempts):
        """Quarantine leases that expired AFTER exhausting retry budget.

        Expired leases under budget are reclaimed by claim_due().
        When the maximum was already reached, repeatedly reclaiming a
        hung event would loop forever; preserve it for investigation.
        """
        with self._connect() as conn:
            cur = conn.execute("""
                UPDATE outbox
                   SET status='dead_letter', lease_token=NULL,
                       lease_until=NULL, last_error='delivery outcome unknown after lease expiry'
                 WHERE status='leased' AND lease_until <= ? AND attempts >= ?
            """, (now, max_attempts))
            return cur.rowcount

    def mark_delivered(self, record):
        """Acknowledge only the CURRENT lease; stale callbacks are ignored."""
        with self._connect() as conn:
            cur = conn.execute("""
                UPDATE outbox SET status='delivered', lease_token=NULL, lease_until=NULL,
                                  last_error=NULL
                WHERE event_id=? AND status='leased' AND lease_token=?
            """, (record.event_id, record.lease_token))
            return cur.rowcount == 1

    def mark_failure(self, record, *, reason, retryable, policy, now, jitter_fraction):
        """Schedule retry, or quarantine after exhaustion/permanent failure."""
        dead = not retryable or record.attempts >= policy.max_attempts
        next_time = now + policy.delay(record.attempts, jitter_fraction)
        with self._connect() as conn:
            cur = conn.execute("""
                UPDATE outbox
                   SET status=?, next_attempt_at=?, last_error=?,
                       lease_token=NULL, lease_until=NULL
                 WHERE event_id=? AND status='leased' AND lease_token=?
            """, (
                "dead_letter" if dead else "pending", next_time, str(reason)[:500],
                record.event_id, record.lease_token,
            ))
            return ("dead_letter" if dead else "pending") if cur.rowcount else "stale"

    def release_unsent(self, record, *, now, delay=0.2):
        """Release a local BufferError without counting an accepted send."""
        with self._connect() as conn:
            cur = conn.execute("""
                UPDATE outbox SET status='pending', attempts=attempts-1,
                                  next_attempt_at=?, lease_token=NULL,
                                  lease_until=NULL, last_error='local_queue_full'
                WHERE event_id=? AND status='leased' AND lease_token=?
            """, (now + delay, record.event_id, record.lease_token))
            return cur.rowcount == 1

    def counts(self):
        with self._connect() as conn:
            rows = conn.execute("SELECT status, COUNT(*) AS n FROM outbox GROUP BY status").fetchall()
        return {row["status"]: row["n"] for row in rows}

    def get(self, event_id):
        with self._connect() as conn:
            row = conn.execute("SELECT * FROM outbox WHERE event_id=?", (event_id,)).fetchone()
            return dict(row) if row else None

    def pending_dead_letters(self, limit=20):
        with self._connect() as conn:
            rows = conn.execute("""
                SELECT * FROM outbox
                WHERE status='dead_letter' AND dlq_published=0
                ORDER BY rowid LIMIT ?
            """, (limit,)).fetchall()
            return [dict(row) for row in rows]

    def mark_dlq_published(self, event_id):
        with self._connect() as conn:
            conn.execute("""
                UPDATE outbox SET dlq_published=1
                WHERE event_id=? AND status='dead_letter'
            """, (event_id,))

    def requeue_dead_letter(self, event_id, *, now):
        """Manual operator action after investigating/fixing the cause.

        We keep the SAME stable event_id for consumer-side deduplication.
        """
        with self._connect() as conn:
            cur = conn.execute("""
                UPDATE outbox SET status='pending', attempts=0,
                       next_attempt_at=?, lease_token=NULL, lease_until=NULL,
                       last_error=NULL, dlq_published=0
                WHERE event_id=? AND status='dead_letter'
            """, (now, event_id))
            if cur.rowcount != 1:
                raise ValueError("Event is not in dead-letter status")

## 6.2 Prove the database transaction works

**Objective:** See what survives before Kafka even enters the picture.

- A created order has exactly one matching outbox event.
- If the database rejects the order (duplicate order ID), the outbox insert is also rolled back.
- No topic creation or Kafka call occurs inside the database transaction.

In [ ]:
lab_dir = tempfile.TemporaryDirectory(prefix="kafka_nb5_")
lab_db = Path(lab_dir.name) / "orders.sqlite3"
outbox = SQLiteOutbox(lab_db)

first_event = outbox.create_order(order_id="order-101", customer_id="customer-7",
                                 amount_cents=1299)
print("Persisted event ID:", first_event)
print("Counts:", outbox.counts())
assert outbox.get(first_event)["status"] == "pending"

try:
    outbox.create_order(order_id="order-101", customer_id="customer-7", amount_cents=1299)
except sqlite3.IntegrityError:
    print("Duplicate order rejected; transaction rolled back.")
assert outbox.counts() == {"pending": 1}

---
# Section 7 — Leases and callbacks: the two critical state transitions

**Objective:** Avoid marking a record delivered merely because `produce()` returned successfully.

A relay **claims** a due outbox row. That only reserves it for one worker. After `produce()` queues it, we still wait for the **delivery callback**.

```text
PENDING --claim--> LEASED --success callback--> DELIVERED
                        |
                        +--failed callback--> PENDING (delayed retry)
                        |                     or DEAD_LETTER
                        |
                        +--process crash--> lease expires --> LEASED by new worker
```

**Why token-check a callback?** Suppose worker A claimed an event, then stalled. Its lease expired, so worker B claimed and resent it. If worker A's old callback arrives later, it must **not overwrite** worker B's current state. The lease token makes the old callback stale.

**Important:** Leases protect database claims; they do not make duplicates impossible. The original Kafka message may have succeeded while your process was unable to record success.

In [ ]:
# A small standalone lease experiment: no Kafka required.
lease_clock = FakeClock()
lease_dir = tempfile.TemporaryDirectory(prefix="kafka_lease_demo_")
lease_store = SQLiteOutbox(Path(lease_dir.name) / "lease.sqlite3")
event_id = lease_store.create_order(order_id="order-lease", customer_id="customer-8",
                                    amount_cents=500, now=lease_clock.wall())
first_claim = lease_store.claim_due(now=lease_clock.wall(), lease_seconds=10)
print("First claim attempts:", first_claim.attempts)
print("Before expiry, new claim:", lease_store.claim_due(now=lease_clock.wall()))
lease_clock.advance(11)
second_claim = lease_store.claim_due(now=lease_clock.wall(), lease_seconds=10)
print("After expiry, new claim attempts:", second_claim.attempts)
print("Old callback accepted?", lease_store.mark_delivered(first_claim))
print("New callback accepted?", lease_store.mark_delivered(second_claim))
assert first_claim.lease_token != second_claim.lease_token
assert lease_store.get(event_id)["status"] == "delivered"
lease_dir.cleanup()

---
# Section 8 — A professional *small* resilient relay

**Objective:** Compose the earlier building blocks without stuffing everything into one giant function.

Our `ResilientOutboxRelay` has exactly five responsibilities:

1. **Check admission:** Is the breaker allowing a new delivery attempt?
2. **Claim an event:** Read one due outbox record with a lease.
3. **Submit asynchronously:** Use the **shared producer** from Notebook 3.
4. **Process callback:** Mark confirmed success or schedule retry/dead-letter.
5. **Service events:** Call `poll()`; call `flush()` only during bounded shutdown.

**Ownership boundaries:**

- The application request owns the **business database transaction**.
- The relay owns **claim/retry/circuit decisions**.
- The shared Kafka producer owns **networking and internal retries**.
- The callback owns **recording the final delivery result**.
- A separate operations/recovery workflow owns **dead-letter review and replay**.

For clarity, this notebook uses one relay poller per shared producer. In a larger design with multiple pollers, callback bookkeeping and lifetime ownership must be coordinated explicitly.

In [ ]:
class ResilientOutboxRelay:
    """Publish durable outbox events using one shared Kafka-compatible producer.

    The producer needs methods:
       produce(topic=..., key=..., value=..., on_delivery=...)
       poll(timeout)
       flush(timeout)

    Dependencies are injected so this class can be tested without Kafka.
    """

    def __init__(self, *, store, producer, retry_policy, breaker,
                 now=time.time, random_fraction=random.random,
                 is_retryable=lambda error: True,
                 lease_seconds=60.0):
        self.store = store
        self.producer = producer
        self.retry_policy = retry_policy
        self.breaker = breaker
        self.now = now
        self.random_fraction = random_fraction
        # Classification is a BUSINESS/OPERATIONS policy. Default: retry
        # final failed delivery up to a finite budget, then quarantine.
        self.is_retryable = is_retryable
        self.lease_seconds = lease_seconds
        self._closed = False
        self._lock = threading.Lock()
        self.metrics = {
            "enqueued": 0,
            "acked": 0,
            "failed_reports": 0,
            "queue_full": 0,
            "dead_lettered": 0,
        }

    def _increment(self, name):
        with self._lock:
            self.metrics[name] += 1

    def _delivery_callback(self, record, permit, error, message):
        """Called during producer.poll()/flush(); do not block on network."""
        if error is None:
            accepted = self.store.mark_delivered(record)
            if accepted:
                self._increment("acked")
                self.breaker.success(permit)
            return

        self._increment("failed_reports")
        retryable = bool(self.is_retryable(error))
        result = self.store.mark_failure(
            record, reason=str(error), retryable=retryable,
            policy=self.retry_policy, now=self.now(),
            jitter_fraction=self.random_fraction(),
        )
        if result != "stale":
            if result == "dead_letter":
                self._increment("dead_lettered")
            # A permanently invalid event is not evidence of a sick broker.
            # Only retryable delivery failures contribute to circuit health.
            if retryable:
                self.breaker.failure(permit)
            else:
                self.breaker.cancel(permit)

    def dispatch_once(self, max_events=10):
        """Queue up to max_events DUE outbox events, then service events.

        Returns number accepted into the producer's LOCAL queue.
        This number is NOT the number acknowledged by Kafka.
        """
        if self._closed:
            raise RuntimeError("relay already closed")
        if max_events < 1:
            raise ValueError("max_events must be >= 1")
        accepted_count = 0

        # An event can be stuck with NO callback (crash/hang). Expired
        # leases get reclaimed while under budget; once exhausted,
        # quarantine them instead of retrying forever.
        expired_quarantined = self.store.quarantine_expired(
            now=self.now(), max_attempts=self.retry_policy.max_attempts
        )
        for _ in range(expired_quarantined):
            self._increment("dead_lettered")

        for _ in range(max_events):
            permit = self.breaker.allow()
            if permit is None:
                break  # Circuit OPEN or test probe already in flight.

            record = self.store.claim_due(now=self.now(), lease_seconds=self.lease_seconds)
            if record is None:
                self.breaker.cancel(permit)
                break

            # Bind the event identity to THIS callback; the callback may
            # run long after dispatch_once() returns.
            def report(error, message, record=record, permit=permit):
                self._delivery_callback(record, permit, error, message)

            try:
                self.producer.produce(
                    topic=record.topic,
                    key=record.message_key.encode("utf-8"),
                    value=record.payload_json.encode("utf-8"),
                    on_delivery=report,
                )
            except BufferError:
                # This event was NOT accepted into the local producer queue.
                # Requeue it and keep its application attempt count unchanged.
                self.store.release_unsent(record, now=self.now())
                self.breaker.cancel(permit)
                self._increment("queue_full")
                break
            except Exception as error:
                # Unexpected synchronous errors (e.g. invalid configuration
                # or programming bug) should NOT be retried forever.
                # Quarantine and log; an operator must investigate.
                log.exception("Event %s rejected during enqueue", record.event_id)
                result = self.store.mark_failure(
                    record, reason=f"enqueue error: {error}",
                    retryable=False, policy=self.retry_policy,
                    now=self.now(), jitter_fraction=self.random_fraction(),
                )
                if result == "dead_letter":
                    self._increment("dead_lettered")
                # Not every synchronous error means Kafka is unhealthy.
                self.breaker.cancel(permit)
                break
            else:
                accepted_count += 1
                self._increment("enqueued")

        # Running poll services delivery callbacks; background networking
        # already does the actual sending.
        self.producer.poll(0)
        return accepted_count

    def service_events(self, timeout=0.1):
        """Call regularly in the relay worker loop, even if no new rows exist."""
        return self.producer.poll(timeout)

    def close(self, timeout=10.0):
        """Stop dispatching and allow remaining delivery reports to complete.

        If the timeout expires, leased rows are NOT falsely marked delivered;
        a new worker can recover them after their leases expire.
        """
        self._closed = True
        return self.producer.flush(timeout)


print("Reusable relay class defined. No Kafka connection made.")

## 8.1 Forward quarantined records to a Kafka DLQ (after Kafka recovers)

**Objective:** Make DLQ publishing an actual runnable component—not a magic exception handler.

The database row with `status='dead_letter'` is the durable recovery record. `DeadLetterForwarder` publishes a **copy** to a Kafka DLQ topic and changes `dlq_published` to `1` **only after a successful delivery callback**. If the same cluster is down, it cannot publish, so the database record stays available.

This short forwarder assumes **one DLQ forwarding worker**. Multiple replicas need their own atomic DLQ-claim/lease fields, and the forwarding loop needs scheduling, backpressure and shutdown supervision before production deployment.

In [ ]:
class DeadLetterForwarder:
    # Single-worker example: mirror durable dead letters to a Kafka DLQ topic.

    def __init__(self, *, store, producer, dlq_topic=DLQ_TOPIC):
        self.store = store
        self.producer = producer  # Can reuse the same shared producer.
        self.dlq_topic = dlq_topic
        self.confirmed = 0
        self.failed = 0

    def forward_once(self, max_events=10):
        # Try to enqueue dead-letter copies; confirm using callbacks.
        queued = 0
        for dead in self.store.pending_dead_letters(limit=max_events):
            envelope = {
                "original_event_id": dead["event_id"],
                "original_topic": dead["topic"],
                "attempts": dead["attempts"],
                "reason": dead["last_error"],
                "original_event": json.loads(dead["payload_json"]),
            }

            def on_dlq_delivery(error, message, eid=dead["event_id"]):
                if error is None:
                    self.store.mark_dlq_published(eid)
                    self.confirmed += 1
                else:
                    self.failed += 1
                    log.warning("DLQ copy %s failed: %s", eid, error)

            try:
                self.producer.produce(
                    topic=self.dlq_topic,
                    key=dead["event_id"].encode("utf-8"),
                    value=json.dumps(envelope).encode("utf-8"),
                    on_delivery=on_dlq_delivery,
                )
            except BufferError:
                # Leave the durable row available; retry in a later cycle.
                break
            else:
                queued += 1

        self.producer.poll(0)
        return queued

print("Single-worker DLQ forwarder defined.")

### Read the implementation in five passes

**Pass 1:** `dispatch_once()` checks the breaker *before* claiming database work.

**Pass 2:** `claim_due()` grants a lease; `produce()` accepts a local message but does not confirm delivery.

**Pass 3:** The closure captures the record's `event_id` **and lease token**, so a callback updates the correct row.

**Pass 4:** Successful callback → `delivered`; failed callback → delayed `pending` or `dead_letter`.

**Pass 5:** `close()` flushes only on shutdown. A timed-out flush does **not** erase outbox events.

A real deployment needs a worker scheduler that periodically calls `dispatch_once()` and `service_events()`; it must also handle exceptions, cancellation, monitoring, and its own shutdown deadline. The class is an intentionally small teaching reference, not an enterprise SDK.

---
# Section 9 — Test doubles: simulated Kafka without a broker

**Objective:** Prove the failure paths *deterministically*. Deliberately stopping Docker to test each branch would be slow, flaky, and would mix many different failure modes together.

Our `FakeProducer` will accept a scripted series of outcomes:

| Scripted outcome | Meaning |
|---|---|
| `ok` | Enqueue succeeds; `poll()` invokes successful callback |
| `fail` | Enqueue succeeds; `poll()` invokes failed callback |
| `full` | `produce()` immediately raises `BufferError` |
| `hang` | Enqueue succeeds; no callback arrives (unknown outcome) |

This fake does **not** simulate Kafka's wire protocol, batching, broker retries, or ordering. It only tests **our application state transitions**.

In [ ]:
class FakeDeliveryError(Exception):
    pass


class FakeProducer:
    """Tiny scripted stand-in for Confluent's Producer API."""

    def __init__(self, outcomes):
        self.outcomes = deque(outcomes)
        self.callbacks = deque()
        self.produced = []
        self.poll_calls = 0
        self.flush_calls = 0
        self.completed_reports = 0

    def produce(self, *, topic, key, value, on_delivery):
        result = self.outcomes.popleft() if self.outcomes else "ok"
        if result == "full":
            raise BufferError("Local: Queue full")
        self.produced.append({"topic": topic, "key": key, "value": value})
        if result == "ok":
            self.callbacks.append((on_delivery, None))
        elif result == "fail":
            self.callbacks.append((on_delivery, FakeDeliveryError("broker delivery failed")))
        elif result == "hang":
            pass  # Simulate missing callback / unknown outcome.
        else:
            raise ValueError(f"unknown fake outcome: {result}")

    def poll(self, timeout=0):
        self.poll_calls += 1
        processed = 0
        while self.callbacks:
            callback, error = self.callbacks.popleft()
            callback(error, None)
            processed += 1
        self.completed_reports += processed
        return processed

    def flush(self, timeout=10):
        self.flush_calls += 1
        self.poll(0)
        # For this fake only: queued messages minus completed reports.
        return max(0, len(self.produced) - self.completed_reports)


def new_offline_setup(outcomes, *, max_attempts=3, threshold=3, cooldown=5):
    """Make isolated dependencies for an independently reproducible test."""
    folder = tempfile.TemporaryDirectory(prefix="kafka_nb5_test_")
    clk = FakeClock()
    store = SQLiteOutbox(Path(folder.name) / "outbox.sqlite3")
    producer = FakeProducer(outcomes)
    breaker = CircuitBreaker(threshold=threshold,
                             cooldown_seconds=cooldown, clock=clk.monotonic)
    policy = RetryPolicy(max_attempts=max_attempts, initial_delay_seconds=1,
                         max_delay_seconds=8)
    relay = ResilientOutboxRelay(
        store=store, producer=producer, retry_policy=policy,
        breaker=breaker, now=clk.wall, random_fraction=lambda: 1.0,
        lease_seconds=10,
    )
    return folder, clk, store, producer, breaker, relay

print("Fake producer and isolated test setup ready.")

---
# Section 10 — Offline unit tests (no Kafka or Docker needed)

Run these cells **in order**. Every test constructs a fresh SQLite database and simulated producer. Each test proves one engineering promise.

### Test A — Successful acknowledgement marks the row delivered

**Objective:** The database must not say `delivered` until the callback reports success. The fake calls it during `poll()`.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(["ok"])
try:
    eid = store.create_order(order_id="test-A", customer_id="c1", amount_cents=100,
                             now=clk.wall())
    assert store.get(eid)["status"] == "pending"
    queued = relay.dispatch_once(max_events=1)
    assert queued == 1
    assert store.get(eid)["status"] == "delivered"
    assert relay.metrics["acked"] == 1
    print("PASS A: callback-confirmed delivery persisted")
finally:
    folder.cleanup()

### Test B — Final delivery error schedules a delayed retry

**Objective:** A callback error does not erase the event; retry respects backoff and retains its identity.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(["fail", "ok"])
try:
    eid = store.create_order(order_id="test-B", customer_id="c1", amount_cents=100,
                             now=clk.wall())
    relay.dispatch_once(max_events=1)
    after_failure = store.get(eid)
    assert after_failure["status"] == "pending" and after_failure["attempts"] == 1
    assert relay.dispatch_once(max_events=1) == 0  # Not due yet.
    clk.advance(1.0)
    relay.dispatch_once(max_events=1)
    assert store.get(eid)["status"] == "delivered"
    assert store.get(eid)["attempts"] == 2
    assert all(json.loads(item["value"])["event_id"] == eid for item in fake.produced)
    print("PASS B: delayed retry + stable event ID")
finally:
    folder.cleanup()

### Test C — Local queue full does NOT mean the broker rejected an event

**Objective:** `BufferError` should leave the event safely pending; it should not count as an accepted dispatch or incorrectly open the circuit breaker.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(["full", "ok"])
try:
    eid = store.create_order(order_id="test-C", customer_id="c1", amount_cents=100,
                             now=clk.wall())
    assert relay.dispatch_once(max_events=1) == 0
    row = store.get(eid)
    assert row["status"] == "pending" and row["attempts"] == 0
    assert breaker.state == "CLOSED"
    clk.advance(0.2)
    assert relay.dispatch_once(max_events=1) == 1
    assert store.get(eid)["status"] == "delivered"
    print("PASS C: local queue pressure handled without loss")
finally:
    folder.cleanup()

### Test D — Circuit breaker blocks new dispatch during repeated failures

**Objective:** After failures, leave fresh work in the outbox. A successful probe after cooldown restores normal dispatch.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(
    ["fail", "fail", "ok", "ok", "ok"], threshold=2, cooldown=5
)
try:
    ids = [store.create_order(order_id=f"test-D-{i}", customer_id="c1",
                              amount_cents=100, now=clk.wall()) for i in range(3)]
    # One callback failure per cycle; without waiting for retry deadlines,
    # other fresh rows can be claimed (and the breaker sees the failures).
    assert relay.dispatch_once(max_events=1) == 1
    assert relay.dispatch_once(max_events=1) == 1
    assert breaker.state == "OPEN"
    assert relay.dispatch_once(max_events=1) == 0
    assert sum(store.get(eid)["attempts"] for eid in ids) == 2
    clk.advance(5)
    assert relay.dispatch_once(max_events=1) == 1
    assert breaker.state == "CLOSED"
    print("PASS D: breaker opens, prevents new claims, recovers via probe")
finally:
    folder.cleanup()

### Test E — Exhausted attempts are quarantined, not deleted

**Objective:** When the budget expires, keep the original event and failure metadata available for inspection and manual replay.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(
    ["fail", "fail"], max_attempts=2, threshold=10
)
try:
    eid = store.create_order(order_id="test-E", customer_id="c1", amount_cents=100,
                             now=clk.wall())
    relay.dispatch_once(max_events=1)
    clk.advance(1.0)
    relay.dispatch_once(max_events=1)
    row = store.get(eid)
    assert row["status"] == "dead_letter" and row["attempts"] == 2
    assert row["last_error"]
    assert store.pending_dead_letters()[0]["event_id"] == eid
    print("PASS E: dead-letter row retained with attempt count + cause")
    # Replay is an EXPLICIT operator decision after investigation.
    store.requeue_dead_letter(eid, now=clk.wall())
    assert store.get(eid)["status"] == "pending"
    assert store.get(eid)["attempts"] == 0
    print("PASS E: manual replay restored the same stable event")
finally:
    folder.cleanup()

### Test F — A missing callback or shutdown timeout is an UNKNOWN outcome

**Objective:** Never silently claim success when Kafka's response did not arrive. Recover with an expired lease.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(["hang", "ok"])
try:
    eid = store.create_order(order_id="test-F", customer_id="c1", amount_cents=100,
                             now=clk.wall())
    assert relay.dispatch_once(max_events=1) == 1
    assert store.get(eid)["status"] == "leased"   # Not delivered!
    outstanding = relay.close(timeout=0.01)
    assert outstanding > 0
    assert store.get(eid)["status"] == "leased"
    clk.advance(11)  # Lease expires (simulates restart/new relay).
    replacement = ResilientOutboxRelay(
        store=store, producer=fake, retry_policy=RetryPolicy(max_attempts=3),
        breaker=CircuitBreaker(clock=clk.monotonic),
        now=clk.wall, random_fraction=lambda: 1.0, lease_seconds=10
    )
    replacement.dispatch_once(max_events=1)
    assert store.get(eid)["status"] == "delivered"
    assert store.get(eid)["attempts"] == 2
    print("PASS F: unknown outcome retained and recovered after lease expiry")
finally:
    folder.cleanup()

### Test G — Concurrent relays must not claim the same row together

**Objective:** Demonstrate that the database claim is atomic. This verifies the repository behavior; it does not prove end-to-end exactly-once delivery.

In [ ]:
folder = tempfile.TemporaryDirectory(prefix="kafka_nb5_concurrent_")
try:
    store = SQLiteOutbox(Path(folder.name) / "concurrent.sqlite3")
    eid = store.create_order(order_id="test-G", customer_id="c1", amount_cents=100, now=1000)
    claims = []
    lock = threading.Lock()
    gate = threading.Barrier(4)

    def contender():
        gate.wait()
        record = store.claim_due(now=1000, lease_seconds=30)
        with lock:
            claims.append(record)

    threads = [threading.Thread(target=contender) for _ in range(3)]
    for worker in threads:
        worker.start()
    gate.wait()
    for worker in threads:
        worker.join()
    assert sum(record is not None for record in claims) == 1
    assert store.get(eid)["status"] == "leased"
    print("PASS G: exactly one database lease granted concurrently")
finally:
    folder.cleanup()

### Test H — Permanently invalid delivery is quarantined without opening the breaker

**Objective:** An invalid business message is not evidence of a broker outage. A classifier can send it directly to dead-letter status without repeated retries.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(["fail"], threshold=1)
try:
    eid = store.create_order(order_id="test-H", customer_id="c1", amount_cents=100,
                             now=clk.wall())
    relay.is_retryable = lambda error: False  # Inject a business error classifier.
    relay.dispatch_once(max_events=1)
    assert store.get(eid)["status"] == "dead_letter"
    assert store.get(eid)["attempts"] == 1
    assert breaker.state == "CLOSED"  # Poison message != unavailable broker.
    print("PASS H: permanent error quarantined; breaker remains healthy")
finally:
    folder.cleanup()

### Test I — Repeated missing callbacks cannot loop forever

**Objective:** Even when an event never gets a callback, it is eventually quarantined after a finite number of expired leases, rather than endlessly redispatched.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(
    ["hang", "hang"], max_attempts=2, threshold=5
)
try:
    eid = store.create_order(order_id="test-I", customer_id="c1", amount_cents=100,
                             now=clk.wall())
    assert relay.dispatch_once(max_events=1) == 1
    clk.advance(11)
    assert relay.dispatch_once(max_events=1) == 1
    clk.advance(11)
    assert relay.dispatch_once(max_events=1) == 0
    row = store.get(eid)
    assert row["status"] == "dead_letter" and row["attempts"] == 2
    assert "unknown" in row["last_error"]
    print("PASS I: exhausted hung deliveries retained for investigation")
finally:
    folder.cleanup()

### Test J — Transactional outbox survives a failed business write

**Objective:** If an order insert is rejected, the outbox must not contain an orphan event for that transaction.

In [ ]:
folder = tempfile.TemporaryDirectory(prefix="kafka_nb5_atomic_")
try:
    store = SQLiteOutbox(Path(folder.name) / "atomic.sqlite3")
    store.create_order(order_id="test-J", customer_id="c1", amount_cents=100)
    before = store.counts().copy()
    try:
        store.create_order(order_id="test-J", customer_id="c2", amount_cents=200)
        assert False, "Expected duplicate order error"
    except sqlite3.IntegrityError:
        pass
    assert store.counts() == before
    print("PASS J: business-order failure rolled back its event")
finally:
    folder.cleanup()

### Test K — DLQ copies are marked published only after their own acknowledgement

**Objective:** Publishing a DLQ copy is another asynchronous Kafka send, with its own callback and possible failure. The underlying dead-letter record is never lost.

In [ ]:
folder, clk, store, fake, breaker, relay = new_offline_setup(["fail"], max_attempts=1)
try:
    eid = store.create_order(order_id="test-K", customer_id="c1", amount_cents=100,
                             now=clk.wall())
    relay.dispatch_once(max_events=1)
    assert store.get(eid)["status"] == "dead_letter"
    dlq_producer = FakeProducer(["fail", "ok"])
    forwarder = DeadLetterForwarder(store=store, producer=dlq_producer)
    assert forwarder.forward_once() == 1
    assert store.get(eid)["dlq_published"] == 0
    assert forwarder.forward_once() == 1
    assert store.get(eid)["dlq_published"] == 1
    assert forwarder.confirmed == 1 and forwarder.failed == 1
    assert len(dlq_producer.produced) == 2
    assert all(item["topic"] == DLQ_TOPIC for item in dlq_producer.produced)
    print("PASS K: DLQ copy needs separate successful acknowledgement")
finally:
    folder.cleanup()

### Offline test results: what have we proved?

Our own code correctly handles **acknowledgements, failed reports, local queue saturation, circuit-breaking, retry exhaustion, lease recovery, and concurrent claims** under simulated outcomes.

We have **not** proved real cluster throughput, broker durability, security configuration, exactly-once behavior, or failure recovery across actual machines. Those require Kafka integration tests, chaos tests, and operational monitoring.

---
# Section 11 — Optional LIVE Kafka integration

**Objective:** See an outbox event move from `pending` to `delivered` using the real Confluent Python client.

**Requirements:** `confluent-kafka` installed; broker reachable; permission to create/write to two lab topics. **No topic is deleted or recreated.** Live steps remain off unless you explicitly set `RUN_LIVE_KAFKA = True` in Section 0 and rerun this cell.

**Caution:** Only run the live example in a **development/test cluster**, never against production topics. With one broker and replication factor 1, `acks='all'` is not the same as multi-broker durability.

In [ ]:
if RUN_LIVE_KAFKA:
    from confluent_kafka import Producer
    from confluent_kafka.admin import AdminClient, NewTopic

    admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
    existing = set(admin.list_topics(timeout=10).topics)

    # Only create missing LAB topics; never delete existing topics.
    for topic in (ORDERS_TOPIC, DLQ_TOPIC):
        if topic not in existing:
            future = admin.create_topics([
                NewTopic(topic, num_partitions=3, replication_factor=1)
            ])[topic]
            future.result(timeout=15)
            print("Created lab topic:", topic)

    live_producer = Producer({
        "bootstrap.servers": BOOTSTRAP,
        "acks": "all",
        "enable.idempotence": True,
        "delivery.timeout.ms": 15000,
    })

    live_folder = tempfile.TemporaryDirectory(prefix="kafka_nb5_live_")
    live_store = SQLiteOutbox(Path(live_folder.name) / "outbox.sqlite3")
    live_event_id = live_store.create_order(
        order_id="lab5-" + uuid.uuid4().hex[:8],
        customer_id="lab-customer", amount_cents=2599,
    )

    live_relay = ResilientOutboxRelay(
        store=live_store,
        producer=live_producer,
        retry_policy=RetryPolicy(max_attempts=3, initial_delay_seconds=1),
        breaker=CircuitBreaker(threshold=3, cooldown_seconds=5),
        lease_seconds=60,
    )

    deadline = time.monotonic() + 20
    try:
        while time.monotonic() < deadline:
            live_relay.dispatch_once(max_events=5)
            live_relay.service_events(0.25)
            status = live_store.get(live_event_id)["status"]
            if status in ("delivered", "dead_letter"):
                break
        print("Live event status:", live_store.get(live_event_id)["status"])
        print("Live metrics:", live_relay.metrics)
    finally:
        remaining = live_relay.close(timeout=16)
        print("Remaining producer work after shutdown:", remaining)
        live_folder.cleanup()
else:
    print("Skipped LIVE Kafka lab. Set RUN_LIVE_KAFKA=True in Section 0 to enable.")

## 11.1 Kafka DLQ projection: use the actual forwarder

**Objective:** After recovery, mirror durable `dead_letter` records into the Kafka DLQ topic.

The **runnable** `DeadLetterForwarder` was defined in Section 8.1 and tested in Test K. With a reachable broker, use the same `Producer` instance to send a quarantined event to `DLQ_TOPIC`:

```python
# Illustrative live-worker invocation (requires a real Kafka Producer):
# forwarder = DeadLetterForwarder(store=live_store, producer=live_producer)
# forwarder.forward_once(max_events=10)
# live_producer.poll(0.2)  # service asynchronous DLQ delivery callbacks
```

- A successful DLQ **delivery callback** sets `dlq_published=1` in the database.
- A failed DLQ publish leaves `dlq_published=0` so another cycle can retry.
- If the broker remains unreachable, the source database still contains the dead-letter record.
- Retrying after an ambiguous outcome may create duplicate Kafka DLQ messages; deduplicate with `original_event_id`.

**Before multi-replica production:** Add an atomic DLQ forwarding claim/lease, retry delay, maximum in-flight records, secure payload redaction, and alerting. Without these, two workers can forward the same record simultaneously.

---
# Section 12 — Production integration: where this lives in FastAPI

**Objective:** Assign lifecycle ownership correctly.

A scalable service should **not** have each HTTP endpoint create a Kafka Producer or block until the broker is reachable. The request persists the business change + outbox event transactionally; a **separate relay worker** owns publishing.

```text
                         FASTAPI WEB PROCESS
POST /orders --> validate --> DB transaction --> HTTP success
                                    |
                                    v
                         Orders + Outbox (durable)
                                    |
                                  poll due
                                    v
                         BACKGROUND WORKER PROCESS
                         +-------------------------+
                         | SQLite/Postgres outbox  |
                         | Circuit breaker         |
                         | Shared Kafka Producer   |
                         | Callback + retry        |
                         +-------------------------+
                                    |
                                    v
                              Kafka cluster
```

**Very important:** An HTTP `201 Created` after the database transaction means the **order and its outbox event were persisted**, not that Kafka has acknowledged the message. If your API contract requires confirmation of Kafka delivery before returning, that is a different latency/availability trade-off.

### A small FastAPI application sketch

```python
# API startup: initialize DB repository, not a Producer per request.
# Request handler: store order + outbox record transactionally.
# Separate worker startup: initialize one producer and one relay.
# Worker loop: dispatch_once(); service_events(timeout=...)
# Worker shutdown: close(timeout=...)
# DB row leases: reclaim events after crashed workers.
```

**Worker scheduler in production:** Implement concurrency-safe worker ownership, error supervision, metrics, controlled shutdown, and row claim fairness. In Azure, this could be a separately deployed Azure Container Apps worker using an application SQL database; there is no need to tie Kafka publishing to a specific cloud provider.

### Why not use `FastAPI.BackgroundTasks` for guaranteed delivery?

FastAPI background tasks execute in the application process and may be lost if that process crashes or is redeployed. They can be useful to *signal* or *accelerate* the outbox worker, but **the durable outbox remains the recovery mechanism**. A separate worker is generally easier to operate and scale.

---
# Section 13 — Observability: what to measure

**Objective:** Know whether events are delayed or stranded **before** customers complain.

| Metric | What it tells you | Example alert |
|---|---|---|
| Pending outbox count | Amount of work waiting | Backlog grows continually |
| Oldest pending event age | Business latency | Oldest unconfirmed order > SLA |
| Delivery success/failure rate | Kafka outcome trend | Sustained high callback error rate |
| Circuit state | Whether submissions are being paused | Circuit remains OPEN unusually long |
| Local queue-full count | Producer backpressure | Frequent `BufferError` |
| Dead-letter count | Records requiring intervention | New DLQ events without an owner |
| Lease recovery count | Crashes/hung deliveries | Repeated lease expiry |
| DLQ forwarding lag | Quarantined events awaiting copy to Kafka | Stale unsent DLQ projection |

**Log fields:** `event_id`, `order_id` (if permitted), `original_topic`, `attempt_number`, `reason_code`, `elapsed_ms`, `trace_id`. Avoid logging entire sensitive payloads. Monitoring a failing DLQ without an investigation/replay process is not a complete solution.

---
# Section 14 — Production design boundaries engineers must know

1. **At-least-once is expected.** A broker may accept an event before your application records its acknowledgement. On recovery, the outbox may republish it. Consumers should deduplicate by stable `event_id`.
2. **Producer idempotence is not a universal exactly-once guarantee.** It handles relevant producer-session retries; replaying a record from a new producer session can still create a duplicate.
3. **The database must be durable and shared.** SQLite is only a local tutorial. An ephemeral container volume is not a resilient outbox.
4. **Leases have time assumptions.** Too-short leases cause premature retries; use a sufficiently long lease or renew it, and assess clock skew across workers.
5. **Timeout is not proof of failure.** A callback timeout or shutdown timeout can leave the true broker outcome uncertain.
6. **DLQ topic availability is not independent of the cluster.** When Kafka is down, retain failures in a separate durable store; publish a DLQ copy only once the cluster is available.
7. **Failures have different remedies.** Queue pressure → backpressure. Auth/config issues → operator fix. Poison data → validate/quarantine. Broker outages → delay/retry and circuit break.
8. **No infinite retry loops.** Use bounded application retries, a recovery owner, and alarms. Choose the maximum retry age/attempts according to business consequences.
9. **One shared producer per process/configuration is a useful default.** It is thread-safe, but design one clear poll/shutdown owner.
10. **Production hardening needs more than notebook code.** Add schema governance, database migrations, secret management, retention, privacy, tracing, alerting, safe manual replay, worker election, and integration/chaos tests.

### Decisions you should make explicitly

| Scenario | Responsible choice |
|---|---|
| Kafka is briefly slow | Let Kafka's producer retry internally; monitor latency |
| `produce()` raises `BufferError` | Back off and retry local submission with a deadline |
| Delivery callback finally fails | Keep durable event; apply finite application-level replay policy |
| Broker is unavailable for hours | Accumulate safely in outbox; alert on age/size; apply backpressure on incoming writes when necessary |
| Event is permanently invalid | Quarantine; don't overwhelm Kafka with identical invalid retries |
| Retry budget exhausted | Preserve in dead-letter storage; investigate and replay deliberately |
| No callback before shutdown | Leave row leased/pending for recovery; never mark confirmed success |

---
# Section 15 — WHAT ENGINEERS ACTUALLY REMEMBER (one-page cheat sheet)

## The six production rules

1. **`produce()` only accepts the event locally.** Kafka delivery is asynchronous.
2. **`on_delivery` is the final attempt result.** `poll()` and `flush()` run callback code; background networking sends.
3. **Retries need a budget and backoff.** The producer already performs internal retries; avoid runaway application retries.
4. **Circuit breaker protects the dependency.** When it opens, new work must stay safely stored instead of disappearing.
5. **A Kafka DLQ is still Kafka.** If the cluster is unreachable, a same-cluster DLQ cannot save the record.
6. **A transactional outbox is the durability bridge.** Persist the business change and the event together; replay until acknowledged or quarantined.

## Final mental model

```text
ORDER REQUEST
     |
     v
DB TRANSACTION: order + outbox event
     |
     v
OUTBOX WORKER -- breaker admission
     |
     v
SHARED PRODUCER.produce()
     |
     v
BACKGROUND KAFKA NETWORKING
     |
     v
DELIVERY CALLBACK (via poll / flush)
     |
     +-- success ----------------------> MARK DELIVERED
     |
     +-- temporary failure -----------> BACKOFF + RETRY
     |
     +-- exhausted/invalid -----------> DURABLE DEAD LETTER
     |
     +-- unknown / crash -------------> LEASE EXPIRY + REPLAY
```

**Mnemonic:** **STORE → SEND → CONFIRM → RETRY → QUARANTINE → RECOVER**.

## Six interview-ready answers

- **Why use a callback?** To know the final result of an asynchronous send attempt.
- **What causes `BufferError`?** The local producer cannot enqueue more messages right now.
- **What is backoff?** Increasing delay between application retry attempts, usually with jitter.
- **What is a circuit breaker?** A guard that pauses new attempts after repeated failures and later probes recovery.
- **Can I send to the Kafka DLQ if Kafka is down?** Not to the same unreachable cluster; persist failures somewhere independently durable.
- **Does an outbox give exactly-once delivery?** No; it supports reliable at-least-once publishing, so consumers still need deduplication.

---
# Section 16 — Practice tasks and mastery checklist

**Work through these in this order.** You should be able to explain each step before you add more code.

1. **Beginner:** Draw three arrows: local queue rejection, callback failure, unknown outcome. Explain why they are different.
2. **Beginner:** Modify `RetryPolicy` and inspect delays using deterministic `random_fraction=1`.
3. **Intermediate:** Write a fake scenario `fail → fail → ok`; show the same `event_id` reaches `delivered` with three attempts.
4. **Intermediate:** Write a `permanent` classifier for an intentionally invalid event; verify immediate dead-lettering rather than repeated retries.
5. **Intermediate:** Add a database method measuring age of oldest `pending` event.
6. **Advanced:** Add lease renewal for long in-flight sends; show why stale callbacks must not overwrite newer leases.
7. **Advanced:** Implement a DLQ forwarding worker with bounded queue-full retry, independent DLQ claim/lease, and acknowledged publication tracking.
8. **Production mastery:** Replace SQLite with your service's database and add integration tests for an actual Kafka outage and recovery.

### How to check you've mastered Notebook 5

You are ready when you can clearly explain:

- Why `flush()==0` is not sufficient evidence that all records succeeded.
- Why producer internal retries and your durable outbox retries are separate layers.
- Why a callback timeout can create duplicate business events on replay.
- Why a same-cluster DLQ is not an outage recovery store.
- Why one database transaction is the core of the outbox pattern.
- How you will discover, triage, and replay dead-lettered events in production.

## Official references

- [Confluent Python Client API — Producer](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html)
- [Confluent's reference producer example](https://github.com/confluentinc/confluent-kafka-python/blob/master/examples/producer.py)
- [Confluent Kafka Python client GitHub](https://github.com/confluentinc/confluent-kafka-python)

**Final engineering principle:** Reliable producer design is primarily about **durable ownership of events, explicit failure states, and bounded recovery** — not memorizing Kafka configuration values.